# Week 88, Day 3: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w088d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('junctions.csv', 'w', encoding='utf-8') as f:
    f.write('name,height_m\nMill,40\nWeir,55\n"Church, St Mary",80\nRidge,310\nFarm,120\nWood,150\nLake,60\nMill,45\nIsland,5\nPond,abc\nQuay,8\n')
with open('paths.csv', 'w', encoding='utf-8') as f:
    f.write('from,to,km\nMill,Weir,1.5\nWeir,"Church, St Mary",2.0\nMill,"Church, St Mary",4.0\n"Church, St Mary",Ridge,3.0\nWeir,Farm,2.5\nFarm,Ridge,0.4\nFarm,Wood,1.5\nWood,Lake,2.0\nLake,Mill,3.5\nWeir,Mill,1.6\nLake,Moor,2.0\nWood,Wood,1.0\nQuay,Island,-1\nIsland,Quay,0.8\n')

**Predict before you run:** what will this print?



In [ ]:
import csv
import io

line = 'Weir,"Church, St Mary",2.0'
print(line.split(","))
print(next(csv.reader(io.StringIO(line))))

In [ ]:
import csv
from collections import deque

with open("paths.csv", newline="") as f:
    rows = list(csv.DictReader(f))
neighbours = {}
for row in rows:
    a, b = row["from"], row["to"]
    neighbours.setdefault(a, set()).add(b)
    neighbours.setdefault(b, set()).add(a)

seen, parts = set(), []
for start in sorted(neighbours):
    if start in seen:
        continue
    part, queue = [], deque([start])
    seen.add(start)
    while queue:
        node = queue.popleft()
        part.append(node)
        for nxt in sorted(neighbours[node] - seen):
            seen.add(nxt)
            queue.append(nxt)
    parts.append(sorted(part))
print(sorted(parts, key=len, reverse=True))

> **Key idea.** Read CSV with `csv.DictReader`, never by splitting on commas; check the header first; report problems by line number (the header is line 1), keep going, and keep the first of any duplicates. Separate errors (the data is wrong: unknown references, impossible numbers, contradictions) from warnings (suspicious but possible: very steep paths, duplicate paths, disconnected parts). Find disconnected parts as the connected components of the graph, using breadth-first search. Import only error-free data, replacing the old data in one transaction, and report what changed.

### Exercise: Reading junctions

Write `read_junctions(text)`: parse CSV text with header `name,height_m`. Return `(heights, problems)`: `heights` maps each valid junction's name (stripped) to its height; `problems` lists, in line order, `"line N: missing name"`, `"line N: height must be a whole number"` and `"line N: duplicate junction NAME"` (keeping the first). If the header isn't exactly those two columns, return `({}, ["line 1: expected columns name,height_m"])`.

In [ ]:
import csv
import io

def read_junctions(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_junctions():\n    \"quotes, duplicates, bad numbers\"\n    heights, problems = read_junctions(open(\"junctions.csv\").read())\n    assert heights == {\"Mill\": 40, \"Weir\": 55, \"Church, St Mary\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150,\n                       \"Lake\": 60, \"Island\": 5, \"Quay\": 8}\n    assert problems == [\"line 9: duplicate junction Mill\", \"line 11: height must be a whole number\"]\n    assert read_junctions(\"name,height\\nA,1\\n\") == ({}, [\"line 1: expected columns name,height_m\"])\n    assert read_junctions(\"name,height_m\\n ,4\\n B , 7 \\n\")[0] == {\"B\": 7}\n    assert read_junctions(\"name,height_m\\n ,4\\n\")[1] == [\"line 2: missing name\"]")

### Exercise: Reading paths

Write `read_paths(text, heights)`: parse CSV text with header `from,to,km` (else `([], ["line 1: expected columns from,to,km"])`). Return `(paths, problems)`: `paths` is a list of `(a, b, km)` for valid lines in order; `problems` lists, in line order, the **first** problem found on each bad line, checking in this order: `"line N: unknown junction NAME"` (the first unknown of the two), `"line N: path from NAME to itself"`, `"line N: km must be a positive number"`.

In [ ]:
import csv
import io

def read_paths(text, heights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("HEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church, St Mary\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5, \"Quay\": 8}\n\ndef test_paths():\n    \"references, loops, lengths\"\n    paths, problems = read_paths(open(\"paths.csv\").read(), HEIGHTS)\n    assert problems == [\"line 12: unknown junction Moor\", \"line 13: path from Wood to itself\", \"line 14: km must be a positive number\"]\n    assert len(paths) == 11 and paths[1] == (\"Weir\", \"Church, St Mary\", 2.0) and paths[-1] == (\"Island\", \"Quay\", 0.8)\n    assert read_paths(\"a,b,c\\n\", HEIGHTS) == ([], [\"line 1: expected columns from,to,km\"])\n    assert read_paths(\"from,to,km\\nMill,Weir,far\\n\", HEIGHTS)[1] == [\"line 2: km must be a positive number\"]")

### Exercise: Connected parts

Write `components(names, paths)`: the connected components of the graph whose junctions are `names` (every name is in some component, even with no paths) and whose paths are `(a, b, km)`, each component as a sorted list, the list ordered largest first (ties by first name). Use breadth-first search. Then `disconnected(names, paths)`: every component except the first, flattened into one sorted list.

In [ ]:
from collections import deque

def components(names, paths):
    ___

def disconnected(names, paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_components():\n    \"reachable groups\"\n    names = [\"Mill\", \"Weir\", \"Farm\", \"Island\", \"Quay\", \"Pond\"]\n    paths = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Farm\", 2.5), (\"Island\", \"Quay\", 0.8)]\n    assert components(names, paths) == [[\"Farm\", \"Mill\", \"Weir\"], [\"Island\", \"Quay\"], [\"Pond\"]]\n    assert disconnected(names, paths) == [\"Island\", \"Pond\", \"Quay\"]\n    assert components([\"A\", \"B\"], []) == [[\"A\"], [\"B\"]] and disconnected([\"A\"], []) == []")

### Exercise: Suspicious paths

Write `steep_paths(heights, paths, limit=250)`: the paths whose height difference, in metres per kilometre of path, is more than `limit`, as `(a, b, metres_per_km)` with the rate rounded to a whole number, steepest first. Then `duplicate_paths(paths)`: pairs of junctions joined by more than one path in either direction, as sorted `(a, b)` tuples (with `a < b`), sorted.

In [ ]:
def steep_paths(heights, paths, limit=250):
    ___

def duplicate_paths(paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("HEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Farm\": 120, \"Ridge\": 310, \"Wood\": 150}\n\ndef test_plausible():\n    \"ladders and repeats\"\n    paths = [(\"Mill\", \"Weir\", 1.5), (\"Farm\", \"Ridge\", 0.4), (\"Weir\", \"Farm\", 0.2), (\"Weir\", \"Mill\", 1.6), (\"Farm\", \"Wood\", 1.5)]\n    assert steep_paths(HEIGHTS, paths) == [(\"Farm\", \"Ridge\", 475), (\"Weir\", \"Farm\", 325)]\n    assert steep_paths(HEIGHTS, paths, limit=400) == [(\"Farm\", \"Ridge\", 475)]\n    assert duplicate_paths(paths) == [(\"Mill\", \"Weir\")] and duplicate_paths(paths[:2]) == []")

### Exercise: Replacing the map

Write `import_map(conn, heights, paths)`, given tables `junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL)` and `path (start TEXT, finish TEXT, km REAL)`: in **one** transaction, remove every path and every junction not in `heights`, update the heights of junctions that stay, insert new junctions, and insert all the paths. Return `{"added": [...], "updated": [...], "removed": [...]}`, each a sorted list of junction names (`updated` only where the height changed). If anything fails, the database must be unchanged and the error re-raised.

In [ ]:
import sqlite3

def import_map(conn, heights, paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\ndef _db():\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript('''CREATE TABLE junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL);\n                          CREATE TABLE path (start TEXT, finish TEXT, km REAL CHECK (km > 0));\n                          INSERT INTO junction VALUES ('Mill', 40), ('Weir', 50), ('Old Barn', 90);\n                          INSERT INTO path VALUES ('Mill', 'Weir', 1.5), ('Weir', 'Old Barn', 2.0);''')\n    return conn\n\ndef test_import():\n    \"added, updated, removed, all at once\"\n    conn = _db()\n    report = import_map(conn, {\"Mill\": 40, \"Weir\": 55, \"Farm\": 120}, [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Farm\", 2.5)])\n    assert report == {\"added\": [\"Farm\"], \"updated\": [\"Weir\"], \"removed\": [\"Old Barn\"]}\n    assert conn.execute(\"SELECT * FROM junction ORDER BY name\").fetchall() == [(\"Farm\", 120), (\"Mill\", 40), (\"Weir\", 55)]\n    assert conn.execute(\"SELECT * FROM path ORDER BY start\").fetchall() == [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Farm\", 2.5)]\n\ndef test_failure():\n    \"a bad path leaves the old map\"\n    conn = _db()\n    try:\n        import_map(conn, {\"Mill\": 40, \"Weir\": 55}, [(\"Mill\", \"Weir\", -1.0)])\n    except sqlite3.IntegrityError:\n        pass\n    else:\n        raise AssertionError(\"should fail\")\n    assert conn.execute(\"SELECT COUNT(*) FROM junction\").fetchone() == (3,) and conn.execute(\"SELECT height_m FROM junction WHERE name = 'Weir'\").fetchone() == (50,)")

### Bug hunt: A church in two pieces

Splitting each line on commas breaks the quoted name in two: it prints `['Weir', '"Church', ' St Mary"', '2.0']`. Use the `csv` module so it prints `['Weir', 'Church, St Mary', '2.0']`.

In [ ]:
import csv
import io

text = 'from,to,km\nWeir,"Church, St Mary",2.0\n'
rows = [line.split(",") for line in text.splitlines()[1:]]
print(rows[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"quoting understood\"\n    expect_output(\"['Weir', 'Church, St Mary', '2.0']\")")

### Bug hunt: Every line number one too small

The report counts data rows from 1, but the header is line 1 of the file, so every message points at the line above the problem: it prints `line 2: height must be a whole number`. Count the header, so it prints `line 3: height must be a whole number`.

In [ ]:
import csv
import io

text = "name,height_m\nMill,40\nPond,abc\n"
for number, row in enumerate(csv.DictReader(io.StringIO(text)), start=1):
    if not row["height_m"].isdigit():
        print(f"line {number}: height must be a whole number")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the header is line 1\"\n    expect_output(\"line 3: height must be a whole number\")")

### Puzzle: Check, then import

Arrange the service that imports a map: read both files, check them, refuse with the errors if there are any, otherwise import in one transaction and return what changed with the warnings. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the line number of the third data row in a CSV file with a header
q = ___   # "error" or "warning": a path that climbs 300 m in 0.5 km
r = ___   # "error" or "warning": a path to a junction that isn't in the file
s = ___   # how many components: A-B, C-D, E (with no paths)
t = ___   # metres per km for a path of 0.8 km between heights 100 and 340

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and q == \"warning\" and r == \"error\", \"p, q and r\"\n    assert s == 3 and t == 300, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the map checker

Write `check_map(junctions_text, paths_text, steep_limit=250)` using the given readers and checks. Return a dict: `heights` and `paths` (the valid ones), `errors` (the junction problems followed by the path problems), `warnings` (in this order: `"steep path A to B (N m per km)"` for each steep path, steepest first; `"duplicate path between A and B"` for each duplicate pair; and `"not connected to the rest: A, B"` listing the junctions outside the largest component, if any), `ok` (no errors) and `summary`: `"N junctions, M paths, E errors, W warnings"`.

In [ ]:
import csv
import io
from collections import deque

def read_junctions(text):
    reader = csv.DictReader(io.StringIO(text))
    if reader.fieldnames != ["name", "height_m"]:
        return {}, ["line 1: expected columns name,height_m"]
    heights, problems = {}, []
    for line, row in enumerate(reader, start=2):
        name = (row["name"] or "").strip()
        try:
            height = int((row["height_m"] or "").strip())
        except ValueError:
            problems.append(f"line {line}: height must be a whole number")
            continue
        if not name:
            problems.append(f"line {line}: missing name")
        elif name in heights:
            problems.append(f"line {line}: duplicate junction {name}")
        else:
            heights[name] = height
    return heights, problems

def read_paths(text, heights):
    reader = csv.DictReader(io.StringIO(text))
    if reader.fieldnames != ["from", "to", "km"]:
        return [], ["line 1: expected columns from,to,km"]
    paths, problems = [], []
    for line, row in enumerate(reader, start=2):
        a, b = row["from"].strip(), row["to"].strip()
        unknown = next((n for n in (a, b) if n not in heights), None)
        try:
            km = float(row["km"])
        except ValueError:
            km = 0
        if unknown:
            problems.append(f"line {line}: unknown junction {unknown}")
        elif a == b:
            problems.append(f"line {line}: path from {a} to itself")
        elif km <= 0:
            problems.append(f"line {line}: km must be a positive number")
        else:
            paths.append((a, b, km))
    return paths, problems

def outside_largest(names, paths):
    neighbours = {n: set() for n in names}
    for a, b, _ in paths:
        neighbours[a].add(b)
        neighbours[b].add(a)
    seen, parts = set(), []
    for start in sorted(neighbours):
        if start not in seen:
            seen.add(start)
            queue, part = deque([start]), []
            while queue:
                node = queue.popleft()
                part.append(node)
                for nxt in neighbours[node] - seen:
                    seen.add(nxt)
                    queue.append(nxt)
            parts.append(sorted(part))
    parts.sort(key=lambda p: (-len(p), p[0]))
    return sorted(n for p in parts[1:] for n in p)

def check_map(junctions_text, paths_text, steep_limit=250):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checker():\n    \"the file someone sent\"\n    r = check_map(open(\"junctions.csv\").read(), open(\"paths.csv\").read())\n    assert r[\"errors\"] == [\"line 9: duplicate junction Mill\", \"line 11: height must be a whole number\",\n                           \"line 12: unknown junction Moor\", \"line 13: path from Wood to itself\", \"line 14: km must be a positive number\"]\n    assert r[\"warnings\"] == [\"steep path Farm to Ridge (475 m per km)\", \"duplicate path between Mill and Weir\",\n                             \"not connected to the rest: Island, Quay\"]\n    assert r[\"ok\"] is False and r[\"summary\"] == \"9 junctions, 11 paths, 5 errors, 3 warnings\"\n\ndef test_clean():\n    \"a clean map\"\n    r = check_map(\"name,height_m\\nA,10\\nB,20\\nC,30\\n\", \"from,to,km\\nA,B,1\\nB,C,2\\n\")\n    assert r[\"ok\"] is True and r[\"errors\"] == [] and r[\"warnings\"] == [] and r[\"summary\"] == \"3 junctions, 2 paths, 0 errors, 0 warnings\"\n    assert r[\"heights\"] == {\"A\": 10, \"B\": 20, \"C\": 30} and r[\"paths\"] == [(\"A\", \"B\", 1.0), (\"B\", \"C\", 2.0)]")

## Warm-up

The church's name contains a comma, so the CSV file quotes it. Splitting on commas breaks the name in two; the `csv` module understands the quoting. Open `junctions.csv` and `paths.csv` above: they're the map someone has sent for importing, and they have at least seven problems. Try to spot them before the code does.

## Reading CSV properly

Use `csv.DictReader` (Week 23): it reads the header row and gives each record as a dict, handling quotes, commas inside fields and different line endings. Then:

- **Check the header first.** If the columns aren't what you expect, nothing else in the file can be trusted: report it and stop.
- **Report problems by line number**, counting the header as line 1, so someone can open the file and go straight to it: `line 9: duplicate junction Mill`.
- **Keep going** after a bad line, collecting every problem (as with yesterday's walks), and keep the **first** of any duplicates.
- **Strip** spaces from fields, and parse numbers with `int()` and `float()` inside `try`.

## Errors and warnings

Not every problem means the same thing:

- **Errors** make the map wrong or unusable: a path to a junction that doesn't exist, a path of negative length, a junction listed twice with different heights, a path from a place to itself. The import must not go ahead.
- **Warnings** are suspicious but possible: a path that climbs 190 m in 400 m (a ladder, or a typo?), two paths between the same junctions, part of the map that doesn't connect to the rest (an island with a ferry, or a missing path?). Report them, and let a person decide.

A report with both, and an `ok` that's true only when there are no errors, is the honest answer to "can I import this?".

## Does it all connect?

A **connected component** is a group of junctions all reachable from each other along paths (Week 49's graph traversal). A map with more than one component has places that no route can reach from the rest: Waymark would answer "no route" without saying why. Find components with a breadth-first search from each junction not yet visited; report every component except the largest as a disconnected part.

## Importing safely

An import **replaces** the map: junctions and paths that are in the file are kept, those that aren't are removed. Do it in one transaction (`with conn:`): delete the old paths and junctions, insert the new ones, and if anything fails the database still holds the old map, whole. And only import a map whose check found no errors. Return what changed (junctions added, updated and removed), which is what the person who ran the import wants to know.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Your project's input**: wherever data comes in (a file, a form, an API body), list what could be wrong with it, and sort the list into errors and warnings.
2. **Build today's story**, with a test for at least three of those problems, each checking the exact message.
3. **Make any bulk change transactional**, with a test that a failure halfway leaves the old data.
4. **Merge** and update the burndown.

## Before you go

The build days are done. Tomorrow isn't a build day: it's a **tests** day, for the sprint as a whole, filling the gaps and testing the things that are hardest to test.